# 📊 YouTube Trending Video Performance Analysis
**Author:** Jalaj Kumar | **Tools:** Python · Pandas · Matplotlib · Seaborn  
**Dataset:** US YouTube Trending Videos (Jan 2023 – Jun 2024 · 2,000 records)

---
### Project Objective
Explore YouTube trending video data to uncover patterns in views, engagement, upload timing, and channel dominance — providing actionable insights for content creators.

### Key Questions
1. Which video categories receive the highest number of views?
2. What is the best time to upload videos for maximum engagement?
3. How do likes and comments relate to video views?
4. Which channels dominate trending videos?
5. Which days of the week produce the most trending content?

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Style config
NAVY, TEAL, ORANGE = '#1B3A6B', '#1A7A8A', '#E85D04'
PALETTE = [NAVY, TEAL, ORANGE, '#F4A261', '#2D6A4F',
           '#52B788', '#9B5DE5', '#F72585', '#4895EF', '#FFB703']
sns.set_theme(style='white')
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': NAVY,
})

print('Libraries loaded successfully ✅')

## 1. Load & Explore the Dataset

In [ ]:
# Load dataset
df = pd.read_csv('data/youtube_trending_US.csv', parse_dates=['publish_date', 'trending_date'])

print(f'Shape: {df.shape}')
print(f'Date range: {df.publish_date.min().date()} → {df.publish_date.max().date()}')
df.head()

In [ ]:
# Data types and nulls
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())

In [ ]:
# Descriptive statistics
df[['views', 'likes', 'comment_count', 'engagement_score', 'days_to_trend']].describe().round(2)

In [ ]:
# Category distribution
print('Videos per category:')
print(df['category'].value_counts())

---
## Q1: Which video categories receive the highest number of views?

In [ ]:
cat_stats = (
    df.groupby('category')
    .agg(
        total_views   = ('views', 'sum'),
        avg_views     = ('views', 'mean'),
        video_count   = ('video_id', 'count'),
        avg_engagement= ('engagement_score', 'mean'),
    )
    .sort_values('avg_views', ascending=False)
    .reset_index()
)
cat_stats['avg_views_M'] = (cat_stats['avg_views'] / 1e6).round(2)
cat_stats['total_views_B'] = (cat_stats['total_views'] / 1e9).round(2)

cat_stats[['category', 'avg_views_M', 'total_views_B', 'video_count', 'avg_engagement']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Q1 · Video Categories vs Views', fontsize=15, fontweight='bold', color=NAVY)

sorted_cat = cat_stats.sort_values('avg_views', ascending=True)
colors = [TEAL if v == sorted_cat['avg_views'].max() else NAVY for v in sorted_cat['avg_views']]

axes[0].barh(sorted_cat['category'], sorted_cat['avg_views_M'], color=colors, height=0.65)
axes[0].set_xlabel('Avg Views per Video (Millions)')
axes[0].set_title('Average Views per Video by Category')

axes[1].barh(sorted_cat['category'], sorted_cat['total_views_B'], color=ORANGE, alpha=0.85, height=0.65)
axes[1].set_xlabel('Total Views (Billions)')
axes[1].set_title('Total Views Accumulated by Category')

plt.tight_layout()
plt.show()

print('\n📌 Insight: Music leads average views per video (~6.9M). Entertainment dominates total views due to volume.')

---
## Q2: What is the best time to upload videos for maximum engagement?

In [ ]:
hour_stats = (
    df.groupby('publish_hour')
    .agg(avg_views=('views', 'mean'), avg_engagement=('engagement_score', 'mean'))
    .reset_index()
)

# Peak hours
peak = hour_stats.nlargest(3, 'avg_views')
print('Top 3 peak upload hours by avg views:')
print(peak[['publish_hour', 'avg_views', 'avg_engagement']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Q2 · Best Upload Time for Maximum Engagement', fontsize=15, fontweight='bold', color=NAVY)

bar_colors = [TEAL if row['avg_views'] >= hour_stats['avg_views'].quantile(0.75)
              else (ORANGE if row['avg_views'] <= hour_stats['avg_views'].quantile(0.25) else NAVY)
              for _, row in hour_stats.iterrows()]

axes[0].bar(hour_stats['publish_hour'], hour_stats['avg_views']/1e6, color=bar_colors, width=0.8)
axes[0].axvspan(13.5, 16.5, alpha=0.12, color=TEAL, label='Peak window (2pm–4pm)')
axes[0].set_xticks(range(24))
axes[0].set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Avg Views (Millions)')
axes[0].set_title('Average Views by Upload Hour')
axes[0].legend()

pivot = df.pivot_table(values='engagement_score', index='publish_day_of_week',
                       columns='publish_hour', aggfunc='mean')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex([d for d in day_order if d in pivot.index])
sns.heatmap(pivot, ax=axes[1], cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Avg Engagement Score', 'shrink': 0.7})
axes[1].set_xlabel('Upload Hour')
axes[1].set_ylabel('')
axes[1].set_title('Engagement Heatmap  (Day × Hour)')

plt.tight_layout()
plt.show()
print('\n📌 Insight: Upload between 2pm–4pm on Fridays & Saturdays for best engagement.')

---
## Q3: How do likes and comments relate to video views?

In [ ]:
# Correlation analysis
corr_matrix = df[['views', 'likes', 'comment_count', 'engagement_score']].corr()
print('Correlation Matrix:')
print(corr_matrix.round(3))

r_likes    = df['views'].corr(df['likes'])
r_comments = df['views'].corr(df['comment_count'])
print(f'\nPearson r (Views vs Likes):    {r_likes:.4f}')
print(f'Pearson r (Views vs Comments): {r_comments:.4f}')

In [ ]:
sample = df.sample(500, random_state=42)
cat_color_map = {cat: PALETTE[i] for i, cat in enumerate(df['category'].unique())}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Q3 · Likes & Comments vs Views', fontsize=15, fontweight='bold', color=NAVY)

# Scatter: views vs likes
axes[0].scatter(sample['views']/1e6, sample['likes']/1e3,
                c=[cat_color_map[c] for c in sample['category']], alpha=0.5, s=20)
m, b = np.polyfit(df['views']/1e6, df['likes']/1e3, 1)
xs = np.linspace(df['views'].min()/1e6, df['views'].max()/1e6, 100)
axes[0].plot(xs, m*xs+b, color=ORANGE, lw=2, label=f'r = {r_likes:.2f}')
axes[0].set_xlabel('Views (Millions)'); axes[0].set_ylabel('Likes (Thousands)')
axes[0].set_title('Views vs Likes'); axes[0].legend()

# Scatter: views vs comments
axes[1].scatter(sample['views']/1e6, sample['comment_count']/1e3,
                c=[cat_color_map[c] for c in sample['category']], alpha=0.5, s=20)
m2, b2 = np.polyfit(df['views']/1e6, df['comment_count']/1e3, 1)
axes[1].plot(xs, m2*xs+b2, color=ORANGE, lw=2, label=f'r = {r_comments:.2f}')
axes[1].set_xlabel('Views (Millions)'); axes[1].set_ylabel('Comments (Thousands)')
axes[1].set_title('Views vs Comments'); axes[1].legend()

# Boxplot: engagement by category
cat_order = df.groupby('category')['engagement_score'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='engagement_score', y='category', order=cat_order,
            ax=axes[2], palette=PALETTE)
axes[2].set_xlabel('Engagement Score'); axes[2].set_ylabel('')
axes[2].set_title('Engagement Distribution by Category')

plt.tight_layout()
plt.show()
print(f'\n📌 Insight: Strong correlation views→likes (r={r_likes:.2f}). Comedy & Music punch above their weight in engagement.')

---
## Q4: Which channels dominate trending videos?

In [ ]:
ch_stats = (
    df.groupby(['channel_title', 'category'])
    .agg(
        trending_count = ('video_id', 'count'),
        avg_views      = ('views', 'mean'),
        avg_engagement = ('engagement_score', 'mean'),
    )
    .reset_index()
    .sort_values('trending_count', ascending=False)
    .head(15)
)

ch_stats['avg_views_M'] = (ch_stats['avg_views'] / 1e6).round(2)
ch_stats[['channel_title', 'category', 'trending_count', 'avg_views_M', 'avg_engagement']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Q4 · Channels Dominating Trending', fontsize=15, fontweight='bold', color=NAVY)

sorted_ch = ch_stats.sort_values('trending_count', ascending=True)
axes[0].barh(sorted_ch['channel_title'], sorted_ch['trending_count'],
             color=[cat_color_map.get(c, NAVY) for c in sorted_ch['category']], height=0.7)
axes[0].set_xlabel('Number of Trending Videos')
axes[0].set_title('Top 15 Channels by Trending Count')

bubble_colors = [cat_color_map.get(c, NAVY) for c in ch_stats['category']]
axes[1].scatter(ch_stats['trending_count'], ch_stats['avg_views_M'],
                s=ch_stats['avg_engagement']*8000, c=bubble_colors, alpha=0.75, edgecolors='white')
for _, row in ch_stats.iterrows():
    axes[1].annotate(row['channel_title'], (row['trending_count'], row['avg_views_M']),
                     fontsize=7.5, ha='center', xytext=(0,6), textcoords='offset points')
axes[1].set_xlabel('Trending Video Count')
axes[1].set_ylabel('Avg Views per Video (Millions)')
axes[1].set_title('Volume vs Quality  (bubble = engagement)')

plt.tight_layout()
plt.show()
print('\n📌 Insight: Entertainment channels lead in count; Music channels (BTS, Bad Bunny, Taylor Swift) lead in avg views.')

---
## Q5: Which days of the week produce the most trending content?

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_stats = (
    df.groupby('publish_day_of_week')
    .agg(
        video_count    = ('video_id', 'count'),
        avg_views      = ('views', 'mean'),
        avg_engagement = ('engagement_score', 'mean'),
        avg_days_trend = ('days_to_trend', 'mean'),
    )
    .reindex(day_order)
    .reset_index()
)
print(dow_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Q5 · Day of Week Trending Patterns', fontsize=15, fontweight='bold', color=NAVY)

day_colors = [TEAL if v >= dow_stats['video_count'].quantile(0.6)
              else (ORANGE if v <= dow_stats['video_count'].quantile(0.3) else NAVY)
              for v in dow_stats['video_count']]

axes[0].bar(dow_stats['publish_day_of_week'], dow_stats['video_count'], color=day_colors, width=0.7)
axes[0].set_title('Trending Videos per Day')
axes[0].set_ylabel('Video Count')
axes[0].set_xticklabels(dow_stats['publish_day_of_week'], rotation=30, ha='right')

axes[1].bar(dow_stats['publish_day_of_week'], dow_stats['avg_views']/1e6, color=NAVY, width=0.7)
axes[1].set_title('Avg Views by Upload Day')
axes[1].set_ylabel('Avg Views (Millions)')
axes[1].set_xticklabels(dow_stats['publish_day_of_week'], rotation=30, ha='right')

axes[2].bar(dow_stats['publish_day_of_week'], dow_stats['avg_engagement'], color=ORANGE, alpha=0.9, width=0.7)
axes[2].set_title('Avg Engagement Score by Day')
axes[2].set_ylabel('Engagement Score')
axes[2].set_xticklabels(dow_stats['publish_day_of_week'], rotation=30, ha='right')

plt.tight_layout()
plt.show()
print('\n📌 Insight: Friday & Saturday show highest trending count AND engagement. Wednesday is the weakest day.')

---
## Summary of Key Findings

| # | Question | Finding |
|---|---|---|
| Q1 | Top category by views? | **Music** leads avg views/video (~6.9M); **Entertainment** leads total views |
| Q2 | Best upload time? | **2pm–4pm on Fridays/Saturdays** produce highest engagement |
| Q3 | Likes/comments vs views? | Strong correlation (r≈0.94 likes, r≈0.75 comments) |
| Q4 | Top channels? | Entertainment channels lead count; Music channels lead avg views |
| Q5 | Best day to upload? | **Friday** ranks highest for both volume and engagement |

### Recommendations for Content Creators
1. **Target Music or Entertainment** — these categories consistently attract the highest viewership
2. **Upload on Friday between 2pm–4pm** — maximises both reach and audience engagement
3. **Focus on quality engagement** — high likes-to-view ratios matter beyond raw views
4. **Study top channels** — channels with consistent weekly uploads dominate trending lists